In [27]:
import json 

def load_data(filename):
    """Safely loads and parses the JSON file."""
    with open(filename, "r") as f:
        # We must call json.load() to convert the file text into a Python dictionary
        return json.load(f)

def pages_you_might_like(user_id, data):
    user_pages = {}
    
    # Populate the dictionary with "Super-Safety."
    for user in data.get('users', []):
        # The 'or []' handles the case where liked_pages is explicitly null
        raw_likes = user.get('liked_pages') or []
        user_pages[user['id']] = set(raw_likes)
    
    if user_id not in user_pages:
        return []

    user_liked_pages = user_pages[user_id]
    page_suggestions = {}

    # Compare the user against everyone else
    for other_user_id, other_pages in user_pages.items():
        if other_user_id != user_id:
            # Find common ground
            shared_pages = user_liked_pages.intersection(other_pages)
            
            if shared_pages:
                weight = len(shared_pages) 
                for page in other_pages:
                    if page not in user_liked_pages:
                        page_suggestions[page] = page_suggestions.get(page, 0) + weight
    
    # Sort and return
    sorted_pages = sorted(page_suggestions.items(), key=lambda x: x[1], reverse=True)
    return [page_id for page_id, weight in sorted_pages]

# --- EXECUTION ---
try:
    # Use the cleaned data file you generated earlier
    data = load_data("cleaned_data2.json")
    
    target_user = 1 # Testing with Neo/Amit
    recommendations = pages_you_might_like(target_user, data)
    
    print(f"--- Recommendations for User {target_user} ---")
    if recommendations:
        print(f"Page IDs suggested: {recommendations}")
    else:
        print("No recommendations found. Try liking more pages first!")

except FileNotFoundError:
    print("Error: 'cleaned_data2.json' not found. Make sure the file is in the same folder.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

--- Recommendations for User 1 ---
Page IDs suggested: [109, 102, 103, 104, 105]
